In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from scipy.stats import rankdata
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr, kendalltau, rankdata, entropy
from scipy.spatial.distance import pdist, squareform
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans, DBSCAN
import joblib
import warnings
warnings.filterwarnings('ignore')
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/playground-series-s6e2/sample_submission.csv
/kaggle/input/playground-series-s6e2/train.csv
/kaggle/input/playground-series-s6e2/test.csv
/kaggle/input/pss6e2-artifacts-oofs/optuna_artifacts/metadata.json
/kaggle/input/pss6e2-artifacts-oofs/optuna_artifacts/study_final.pkl
/kaggle/input/pss6e2-artifacts-oofs/optuna_artifacts/all_oof_preds.npy
/kaggle/input/pss6e2-artifacts-oofs/optuna_artifacts/all_models_info.csv
/kaggle/input/pss6e2-artifacts-oofs/optuna_artifacts/all_test_preds.npy
/kaggle/input/pss6e2-artifacts-oofs/optuna_artifacts/trial_oof_preds.pkl
/kaggle/input/pss6e2-artifacts-oofs/optuna_artifacts/trial_predictions.pkl
/kaggle/input/pss6e2-artifacts-oofs2/tabm/skewed_distributions.png
/kaggle/input/pss6e2-artifacts-oofs2/tabm/oof_TabM_D.csv
/kaggle/input/pss6e2-artifacts-oofs2/tabm/submission_TabM_D.csv
/kaggle/input/pss6e2-artifacts-oofs2/tabm/__results___files/__results___12_1.png
/kaggle/input/pss6e2-artifacts-oofs2/tabm/__results___files/__results___17_0.pn

In [2]:
class config:
    SEED = 42
    N_FOLDS = 5
    TARGET = 'Heart Disease'
    
    INPUT_DIR = '/kaggle/input/playground-series-s6e2'

class_mapping = {
    'Presence': 1,
    'Absence': 0
}
rev_class_mapping = {
    0: 'Absence',
    1: 'Presence'
}

CONFIG = config()

In [3]:
train = pd.read_csv('/kaggle/input/playground-series-s6e2/train.csv')
y = train['Heart Disease']
test = pd.read_csv('/kaggle/input/playground-series-s6e2/test.csv')
sample_submission = pd.read_csv('/kaggle/input/playground-series-s6e2/sample_submission.csv')

In [4]:
oofs_gblinear = np.load('/kaggle/input/pss6e2-artifacts-oofs2/gblinear/optuna_artifacts/all_oof_preds.npy')
test_gblinear = np.load('/kaggle/input/pss6e2-artifacts-oofs2/gblinear/optuna_artifacts/all_test_preds.npy')

oofs_gbtree = np.load('/kaggle/input/pss6e2-artifacts-oofs/optuna_artifacts/all_oof_preds.npy')
test_gbtree = np.load('/kaggle/input/pss6e2-artifacts-oofs/optuna_artifacts/all_test_preds.npy')

print(f"gblinear shapes: OOF={oofs_gblinear.shape}, Test={test_gblinear.shape}")
print(f"gbtree shapes: OOF={oofs_gbtree.shape}, Test={test_gbtree.shape}")


oofs_gbtree_df = pd.DataFrame(oofs_gbtree.T)
test_gbtree_df = pd.DataFrame(test_gbtree.T)

oofs_gblinear_df = pd.DataFrame(oofs_gblinear.T)  
test_gblinear_df = pd.DataFrame(test_gblinear.T)

oofs_gblinear_df.columns = [f'gblinear_model_{i}' for i in range(oofs_gblinear_df.shape[1])]
test_gblinear_df.columns = [f'gblinear_model_{i}' for i in range(test_gblinear_df.shape[1])]

oofs_gbtree_df.columns = [f'gbtree_model_{i}' for i in range(oofs_gbtree_df.shape[1])]
test_gbtree_df.columns = [f'gbtree_model_{i}' for i in range(test_gbtree_df.shape[1])]

realmlp_oof = pd.read_csv('/kaggle/input/pss6e2-artifacts-oofs2/realmlp_0.95398/oof.csv').drop(columns='id')
realmlp_test = pd.read_csv('/kaggle/input/pss6e2-artifacts-oofs2/realmlp_0.95398/submission.csv').drop(columns='id')

resnet50_oof = pd.read_csv('/kaggle/input/pss6e2-artifacts-oofs2/resnet50/oof_resnet50.csv').drop(columns='id')
resnet50_test = pd.read_csv('/kaggle/input/pss6e2-artifacts-oofs2/resnet50/submission_resnet50.csv').drop(columns='id')

tabm_oof = pd.read_csv('/kaggle/input/pss6e2-artifacts-oofs2/tabm/oof_TabM_D.csv').drop(columns='id')
tabm_test = pd.read_csv('/kaggle/input/pss6e2-artifacts-oofs2/tabm/submission_TabM_D.csv').drop(columns='id')

cat_oof = pd.read_csv('/kaggle/input/pss6e2-artifacts-oofs2/catboost/oof_original_0.955705176736493.csv').drop(columns='Unnamed: 0').rename(columns={'Heart Disease': 'cat_target'})
cat_test = pd.read_csv('/kaggle/input/pss6e2-artifacts-oofs2/catboost/submission_original_0.955705176736493.csv').drop(columns='id').rename(columns={'Heart Disease': 'cat_target'})

gblinear shapes: OOF=(24, 630000), Test=(24, 270000)
gbtree shapes: OOF=(120, 630000), Test=(120, 270000)


In [5]:
oofs_df = pd.concat([realmlp_oof,  oofs_gbtree_df['gbtree_model_59'], cat_oof], axis=1)
test_df = pd.concat([realmlp_test,  test_gbtree_df['gbtree_model_59'], cat_test], axis=1)

# oofs_df['gblinear_oof'] = oofs_gblinear_df['gblinear_model_23'].values
# test_df['gblinear_test'] = test_gblinear_df['gblinear_model_23'].values

In [6]:
print(f"Combined OOFs shape: {oofs_df.shape}")
print(f"Combined Tests shape: {test_df.shape}")

# print(f"\nModel types distribution:")
# print(f"GBLinear models: {len([c for c in oofs_df.columns if 'gblinear' in c])}")
# print(f"GBTree models: {len([c for c in oofs_df.columns if 'gbtree' in c])}")
# print(f"Total models: {all_oofs_df.shape[1]}")

Combined OOFs shape: (630000, 3)
Combined Tests shape: (270000, 3)


In [7]:
from joblib import Parallel, delayed
from tqdm.auto import tqdm
import time

def fast_auc(y_true, y_prob):
    # Robust Rank-Sum AUC
    desc_score_indices = np.argsort(y_prob)[::-1]
    y_prob = y_prob[desc_score_indices]
    y_true = y_true[desc_score_indices]
    distinct_value_indices = np.where(np.diff(y_prob))[0]
    threshold_idxs = np.r_[distinct_value_indices, y_true.size - 1]
    tps = np.cumsum(y_true)[threshold_idxs]
    fps = 1 + threshold_idxs - tps
    if tps[-1] == 0 or fps[-1] == 0: return 0.5
    return np.trapz(tps / tps[-1], fps / fps[-1])

def hc_worker(X, y, current_weights, std_dev, iterations):
    best_w = current_weights.copy()
    best_score = fast_auc(y, X @ best_w)
    n_models = len(best_w)
    
    for _ in range(iterations):
        # Nudge only 10% of models to find specific improvements
        mask = np.random.rand(n_models) < 0.1
        delta = np.zeros(n_models)
        delta[mask] = np.random.normal(0, std_dev, size=mask.sum())
        
        trial_w = np.maximum(0, best_w + delta)
        trial_w /= (trial_w.sum() + 1e-12)
        
        score = fast_auc(y, X @ trial_w)
        if score > best_score:
            best_score = score
            best_w = trial_w
    return best_w, best_score

def optimized_hill_climb(df_oofs, y_true, n_workers=4, iterations_per_step=10, 
                         patience=15, std_dev=0.005, max_steps=500):
    
    X = df_oofs.values.astype(np.float32)
    y = y_true.values.astype(np.int32) if hasattr(y_true, 'values') else y_true.astype(np.int32)
    
    best_weights = np.ones(X.shape[1]) / X.shape[1]
    best_overall_score = fast_auc(y, X @ best_weights)
    
    print(f"[*] Initial Baseline ROC-AUC: {best_overall_score:.6f}")
    
    bad_steps = 0
    pbar = tqdm(range(max_steps), desc="Optimizing")
    
    for step in pbar:
        # If n_workers is -1, use a safe number like 4 for threading
        num_jobs = 4 if n_workers == -1 else n_workers
        
        results = Parallel(n_jobs=num_jobs, backend="threading")(
            delayed(hc_worker)(X, y, best_weights, std_dev, iterations_per_step)
            for _ in range(num_jobs)
        )
        
        # Fallback if Parallel fails
        if not results:
            res_w, res_s = hc_worker(X, y, best_weights, std_dev, iterations_per_step)
            results = [(res_w, res_s)]

        round_best_w, round_best_score = max(results, key=lambda x: x[1])
        
        if round_best_score > best_overall_score:
            improvement = round_best_score - best_overall_score
            best_overall_score = round_best_score
            best_weights = round_best_w
            bad_steps = 0
        else:
            improvement = 0
            bad_steps += 1
        
        pbar.set_postfix({
            "AUC": f"{best_overall_score:.6f}",
            "Improv": f"{improvement:.2e}",
            "P": f"{bad_steps}/{patience}"
        })
        
        if bad_steps >= patience:
            print(f"\n[!] Early Stopping at step {step}")
            break
            
    return best_weights

y_mapped = y.map({'Presence': 1, 'Absence': 0})

In [8]:
%%time
best_weights = optimized_hill_climb(
    df_oofs=oofs_df, 
    y_true=y_mapped, 
    n_workers=-1,          # Use all available CPU cores
    iterations_per_step=30, # Number of random tries per worker per step
    patience=100,           # How many steps to wait for improvement
    std_dev=0.005,         # Small shifts for 122 models
    max_steps=1000          # Upper limit of generations
)

[*] Initial Baseline ROC-AUC: 0.955768


Optimizing:   0%|          | 0/1000 [00:00<?, ?it/s]


[!] Early Stopping at step 142
CPU times: user 1h 7min 32s, sys: 8 s, total: 1h 7min 40s
Wall time: 17min 13s


In [9]:
best_weights

array([0.34577504, 0.41497263, 0.23925233])

In [10]:
final_preds = test_df.values @ best_weights


In [11]:
# test_preds = test_gbtree_df['gbtree_model_59'].values
sample_submission[CONFIG.TARGET] = final_preds
sample_submission.to_csv('submission_csv_hillclimbing.csv', index=False)

In [12]:
sample_submission

,id,Heart Disease
0,630000,0.950459
1,630001,0.008723
2,630002,0.990325
3,630003,0.004843
4,630004,0.198816
...,...,...
269995,899995,0.144977
269996,899996,0.692782
269997,899997,0.045797
269998,899998,0.172584


In [13]:
# from sklearn.model_selection import StratifiedKFold
# from sklearn.linear_model import Ridge, LogisticRegression
# from sklearn.preprocessing import StandardScaler, LabelEncoder
# from tqdm import tqdm
# import numpy as np

# # Your data
# X_meta_train = all_oofs_df.values
# X_meta_test = all_tests_df.values
# y = train[CONFIG.TARGET].map(class_mapping).values

# strat_cols = ['Thallium', 'Chest pain type', 'Heart Disease']
# le = LabelEncoder()
# stratify_feature = le.fit_transform(train[strat_cols].astype(str).agg('_'.join, axis=1))

# print(f"Meta-train shape: {X_meta_train.shape}")
# print(f"Meta-test shape: {X_meta_test.shape}")

# # Stratified K-Fold setup
# skf = StratifiedKFold(n_splits=CONFIG.N_FOLDS, shuffle=True, random_state=CONFIG.SEED)

# # Storage for predictions
# meta_oof_preds = np.zeros(len(X_meta_train))
# meta_test_preds = np.zeros(len(X_meta_test))
# fold_scores = []

# print(f"\nTraining meta-model with {CONFIG.N_FOLDS}-fold CV")
# print("="*60)

# # CV loop
# for fold, (train_idx, val_idx) in enumerate(skf.split(X_meta_train, stratify_feature), 1):
    
#     print(f"\nFold {fold}:")
#     print(f"Train size: {len(train_idx)}, Val size: {len(val_idx)}")
    
#     # Split meta-data
#     X_train_fold = X_meta_train[train_idx]
#     X_val_fold = X_meta_train[val_idx]
#     y_train_fold = y[train_idx]
#     y_val_fold = y[val_idx]
    
#     # ===== FIXED: USE RIDGE, NOT LOGISTIC (for now) =====
#     # Scale features
#     scaler = StandardScaler()
#     X_train_scaled = scaler.fit_transform(X_train_fold)
#     X_val_scaled = scaler.transform(X_val_fold)
#     X_test_scaled = scaler.transform(X_meta_test)
    
#     # ===== OPTION A: RIDGE REGRESSION (WORKS) =====
#     ridge = Ridge(alpha=0.01, random_state=CONFIG.SEED + fold)
#     ridge.fit(X_train_scaled, y_train_fold)
    
#     # Predict
#     val_preds = ridge.predict(X_val_scaled)
#     test_preds = ridge.predict(X_test_scaled)
    
#     # ===== OPTION B: FIXED LOGISTIC REGRESSION =====
#     # Uncomment this if you want to try logistic
#     # logreg = LogisticRegression(
#     #     C=0.01,  # STRONG REGULARIZATION (1/alpha)
#     #     penalty='l2',
#     #     solver='lbfgs',  # BETTER FOR LARGE DATASETS
#     #     max_iter=1000,
#     #     random_state=CONFIG.SEED + fold
#     # )
#     # logreg.fit(X_train_scaled, y_train_fold)
#     # val_preds = logreg.predict_proba(X_val_scaled)[:, 1]
#     # test_preds = logreg.predict_proba(X_test_scaled)[:, 1]
    
#     # Clip predictions to [0, 1]
#     val_preds = np.clip(val_preds, 0, 1)
#     test_preds = np.clip(test_preds, 0, 1)
    
#     # Check for weird predictions
#     print(f"  Val preds range: [{val_preds.min():.4f}, {val_preds.max():.4f}]")
#     print(f"  Mean val pred: {val_preds.mean():.4f}")
    
#     # Store predictions
#     meta_oof_preds[val_idx] = val_preds
#     meta_test_preds += test_preds / CONFIG.N_FOLDS
    
#     # Score
#     fold_score = roc_auc_score(y_val_fold, val_preds)
#     fold_scores.append(fold_score)
    
#     print(f"  Fold {fold} AUC: {fold_score:.6f}")
#     print(f"  Ridge coef stats: mean={ridge.coef_.mean():.6f}, std={ridge.coef_.std():.6f}")

# # ===== FINAL RESULTS =====
# print(f"\n{'='*60}")
# print("META-MODEL CV RESULTS")
# print(f"{'='*60}")

# print(f"Fold scores: {[f'{s:.6f}' for s in fold_scores]}")
# print(f"Mean fold score: {np.mean(fold_scores):.6f} (±{np.std(fold_scores):.6f})")

# # Check for weird predictions in final OOF
# print(f"\nFinal OOF predictions analysis:")
# print(f"  Range: [{meta_oof_preds.min():.6f}, {meta_oof_preds.max():.6f}]")
# print(f"  Mean: {meta_oof_preds.mean():.6f}")
# print(f"  Std: {meta_oof_preds.std():.6f}")

# # Final score
# final_score = roc_auc_score(y, meta_oof_preds)
# print(f"\nOOF AUC Score: {final_score:.6f}")

# # Compare with simple average
# simple_avg_preds = np.mean(X_meta_train, axis=1)
# simple_avg_score = roc_auc_score(y, simple_avg_preds)
# print(f"Simple average baseline: {simple_avg_score:.6f}")
# print(f"Meta-model improvement: +{final_score - simple_avg_score:.6f}")

# # Create submission
# sample_submission['Heart Disease'] = meta_test_preds
# sample_submission.to_csv(f'submission_meta_ridge_{final_score:.6f}.csv', index=False)
# print(f"\n✅ Saved: submission_meta_ridge_{final_score:.6f}.csv")